# ViperX hand-in-eye auto capture

This notebook is a ViperX version copied from `hand_in_eye_shooting.ipynb`. It keeps the Re3Sim calibration data format, but replaces Franka/Panda-specific robot control with explicit ViperX/LeRobot adapter functions.

The output dataset is still:

```text
base_dir/
  rgb/<frame>.png
  depth/<frame>.npy
  joints/joints_<frame>.npy
  rgb_intrinsics.npz
  depth_intrinsics.npz
```

The active workflow is joint-space automatic capture: send each `q_targets_rad` command with LeRobot, wait for the arm to settle, capture RGB/depth, read measured joints, and save the frame.


# Original Franka/Panda reference, commented out

The next cell keeps the key Franka/Panda calls from the original notebook for reference only. Do not run it for ViperX. The active ViperX interfaces start after this reference cell.


In [ ]:
# Original Franka/Panda reference only. Do not execute for ViperX.
#
# import logging
# import panda_py
# from panda_py import libfranka, constants
# from scipy.spatial.transform import Rotation as R
#
# hostname = "172.16.0.2"
# panda = panda_py.Panda(hostname)
# panda.enable_logging(int(1e2))
# gripper = libfranka.Gripper(hostname)
#
# init_pose = panda_py.fk(constants.JOINT_POSITION_START)
#
# def generate_and_move_to_pose(init_pose, roll, pitch, yaw, z_add, x_add, y_add,
#                               max_roll_deviation, max_pitch_deviation, max_yaw_deviation):
#     roll_turbulent = roll + np.random.uniform(-max_roll_deviation, max_roll_deviation)
#     pitch_turbulent = pitch + np.random.uniform(-max_pitch_deviation, max_pitch_deviation)
#     yaw_turbulent = yaw + np.random.uniform(-max_yaw_deviation, max_yaw_deviation)
#     rotation_matrix = R.from_euler("xyz", [roll_turbulent, pitch_turbulent, yaw_turbulent]).as_matrix()
#
#     pose = init_pose.copy()
#     pose[:3, :3] = init_pose[:3, :3] @ rotation_matrix
#     pose[2, 3] += z_add
#     pose[0, 3] += x_add
#     pose[1, 3] += y_add
#
#     # Franka-specific Cartesian motion. ViperX replacement is joint-space action below.
#     panda.move_to_pose(pose)
#     return pose
#
# # Franka-specific joint readback.
# last_qpos = panda.get_log()["q"][-1]
#
# # Franka-specific return-to-start.
# panda.move_to_start()


# 1. Imports and paths

This section only imports common packages and configures local paths. LeRobot and RealSense connection happens later.


In [ ]:
import os
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np

# Make the local calibration RealSense wrapper importable from this notebook location.
parent_dir = os.path.dirname(os.getcwd())
parent_dir = os.path.dirname(parent_dir)
sys.path.append(parent_dir)

from realsense.realsense import Camera, get_devices

# LeRobot ViperX hardware interface.
# If this import fails, run the notebook from an environment where your local LeRobot repo is installed,
# or add D:/Study/Github/lerobot/src to PYTHONPATH before importing.
from lerobot.robots.viperx import ViperX, ViperXConfig


# 2. Configuration

This section defines the output directory, ViperX connection settings, RealSense resolution, and the automatic capture joint targets. The `q_targets_rad` convention is fixed as radians in the joint order defined by `ViperX_ARM_JOINTS`.


In [ ]:
# Output folder consumed by hand-eye calibration scripts.
base_dir = Path("../../data/hand_in_eye_viperx")

# ViperX serial port. Replace with your actual Dynamixel/U2D2 port.
viperx_port = "COM3"

# RealSense stream settings. Intrinsics are saved from this exact stream configuration.
rgb_resolution = (640, 480)    # (width, height)
depth_resolution = (640, 480)  # (width, height)
delay_before_shooting = 3.0
settle_time_s = 0.5

# Fixed arm joint order used by this notebook.
# Input/output convention for q_rad arrays:
#   shape: (6,)
#   unit: radians
#   order: ViperX_ARM_JOINTS below
ViperX_ARM_JOINTS = [
    "waist",
    "shoulder",
    "elbow",
    "forearm_roll",
    "wrist_angle",
    "wrist_rotate",
]

# Example target list. Replace these with safe ViperX calibration poses.
# Each row is one target joint configuration in radians, ordered by ViperX_ARM_JOINTS.
q_home_rad = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=float)
q_targets_rad = np.array([
    q_home_rad + np.array([0.00,  0.00,  0.00,  0.00,  0.00,  0.00]),
    q_home_rad + np.array([0.20,  0.10, -0.15,  0.15,  0.10, -0.15]),
    q_home_rad + np.array([-0.20, 0.12, -0.10, -0.15,  0.15,  0.15]),
    q_home_rad + np.array([0.10, -0.08,  0.12,  0.20, -0.10,  0.10]),
], dtype=float)

for subdir in ["rgb", "depth", "joints"]:
    (base_dir / subdir).mkdir(parents=True, exist_ok=True)


# 3. ViperX interface layer

This is the replacement boundary for Franka/Panda control. The capture loop below calls only these functions, so LeRobot details stay isolated here.

Interface contract:

- `read_viperx_joints_rad(robot) -> np.ndarray`: returns shape `(6,)`, unit radians, ordered by `ViperX_ARM_JOINTS`.
- `q_rad_to_lerobot_action(q_rad) -> dict[str, float]`: takes shape `(6,)` radians and returns the action dict expected by `robot.send_action()`.
- `move_to_joint_target(...) -> np.ndarray`: takes a target shape `(6,)` radians and returns measured joints shape `(6,)` radians after the move.

The two conversion functions are intentionally placeholders until the LeRobot position to physical joint angle mapping is defined for your robot.


In [ ]:
def create_viperx_robot(port: str) -> ViperX:
    """Create an unconnected LeRobot ViperX object.

    Input:
        port: serial port used by the ViperX Dynamixel bus, e.g. "COM3" or "/dev/ttyDXL".

    Output:
        robot: ViperX instance. Call ``robot.connect()`` before reading or sending actions.
    """
    config = ViperXConfig(port=port, cameras={})
    return ViperX(config)


def read_viperx_joints_rad(robot: ViperX) -> np.ndarray:
    """Read measured ViperX arm joints in radians.

    Input:
        robot: connected LeRobot ViperX instance.

    Output:
        q_rad: np.ndarray with shape (6,), unit radians, ordered by ViperX_ARM_JOINTS.

    Implementation boundary:
        LeRobot ``get_observation()`` returns keys like ``"waist.pos"``. In this ViperX
        driver those values are LeRobot motor positions, not guaranteed to be radians.
        Fill ``lerobot_pos_to_rad`` with your actual mapping, then this function becomes
        the single source of truth for saved calibration joints.
    """
    obs = robot.get_observation()
    q_lerobot = np.array([obs[f"{name}.pos"] for name in ViperX_ARM_JOINTS], dtype=float)
    return lerobot_pos_to_rad(q_lerobot)


def lerobot_pos_to_rad(q_lerobot: np.ndarray) -> np.ndarray:
    """Convert LeRobot ViperX motor positions to radians.

    Input:
        q_lerobot: np.ndarray with shape (6,), ordered by ViperX_ARM_JOINTS.

    Output:
        q_rad: np.ndarray with shape (6,), unit radians, same order.

    Replace this placeholder with your calibrated conversion.
    """
    raise NotImplementedError("Define LeRobot .pos -> ViperX joint radians mapping here.")


def rad_to_lerobot_pos(q_rad: np.ndarray) -> np.ndarray:
    """Convert radians to LeRobot ViperX motor positions.

    Input:
        q_rad: np.ndarray with shape (6,), unit radians, ordered by ViperX_ARM_JOINTS.

    Output:
        q_lerobot: np.ndarray with shape (6,), values accepted by LeRobot send_action().

    Replace this placeholder with the inverse of ``lerobot_pos_to_rad``.
    """
    raise NotImplementedError("Define ViperX joint radians -> LeRobot .pos mapping here.")


def q_rad_to_lerobot_action(q_rad: np.ndarray) -> dict[str, float]:
    """Build a LeRobot action dict from a ViperX joint vector in radians.

    Input:
        q_rad: np.ndarray with shape (6,), unit radians, ordered by ViperX_ARM_JOINTS.

    Output:
        action: dict accepted by ``robot.send_action``. Example keys:
            ``{"waist.pos": value, "shoulder.pos": value, ...}``
    """
    q_lerobot = rad_to_lerobot_pos(np.asarray(q_rad, dtype=float))
    return {f"{name}.pos": float(value) for name, value in zip(ViperX_ARM_JOINTS, q_lerobot)}


def move_to_joint_target(
    robot: ViperX,
    q_target_rad: np.ndarray,
    *,
    settle_time_s: float = 0.5,
    timeout_s: float = 10.0,
    position_tolerance_rad: float = 0.03,
) -> np.ndarray:
    """Command ViperX to a target joint configuration and return measured joints.

    Input:
        robot: connected LeRobot ViperX instance.
        q_target_rad: target joints, shape (6,), radians, ordered by ViperX_ARM_JOINTS.
        settle_time_s: extra wait after sending the command.
        timeout_s: maximum wait for measured joints to approach the target.
        position_tolerance_rad: stop waiting once max absolute joint error is below this.

    Output:
        q_measured_rad: measured joints after the move, shape (6,), radians.
    """
    action = q_rad_to_lerobot_action(q_target_rad)
    robot.send_action(action)
    time.sleep(settle_time_s)

    deadline = time.monotonic() + timeout_s
    q_measured_rad = read_viperx_joints_rad(robot)
    while time.monotonic() < deadline:
        q_measured_rad = read_viperx_joints_rad(robot)
        if np.max(np.abs(q_measured_rad - q_target_rad)) <= position_tolerance_rad:
            break
        time.sleep(0.05)

    return q_measured_rad


# 4. Save helpers

These helpers keep the file format expected by the Re3Sim calibration reader. Saved `joints` are ViperX arm joints in radians, ordered by `ViperX_ARM_JOINTS`.


In [ ]:
def save_rgb(rgb_image: np.ndarray, base_dir: Path, frame_num: int) -> Path:
    """Save one RGB image.

    Input:
        rgb_image: HxWx3 RGB image.
        base_dir: calibration dataset root.
        frame_num: integer frame index.

    Output:
        path: saved PNG path.
    """
    path = base_dir / "rgb" / f"{frame_num}.png"
    plt.imsave(path, rgb_image)
    return path


def save_depth(depth_image: np.ndarray, base_dir: Path, frame_num: int) -> Path:
    """Save one depth image as raw NumPy array.

    Input:
        depth_image: HxW depth array from RealSense wrapper.
        base_dir: calibration dataset root.
        frame_num: integer frame index.

    Output:
        path: saved NPY path.
    """
    path = base_dir / "depth" / f"{frame_num}.npy"
    np.save(path, depth_image)
    return path


def save_joints(q_rad: np.ndarray, base_dir: Path, frame_num: int) -> Path:
    """Save measured ViperX arm joints for one captured frame.

    Input:
        q_rad: np.ndarray with shape (6,), radians, ordered by ViperX_ARM_JOINTS.
        base_dir: calibration dataset root.
        frame_num: integer frame index.

    Output:
        path: saved NPY path.
    """
    path = base_dir / "joints" / f"joints_{frame_num}.npy"
    np.save(path, np.asarray(q_rad, dtype=float))
    return path


def save_camera_intrinsics(camera: Camera, base_dir: Path) -> None:
    """Save RGB/depth intrinsics using the same format as the original notebook.

    Input:
        camera: started RealSense Camera wrapper.
        base_dir: calibration dataset root.

    Output:
        Files written:
            ``rgb_intrinsics.npz``
            ``depth_intrinsics.npz``
    """
    rgb_intrinsics, rgb_coeffs, depth_intrinsics, depth_coeffs = camera.get_intrinsics_raw()
    depth_scale = camera.get_depth_scale()

    np.savez(
        base_dir / "rgb_intrinsics.npz",
        fx=rgb_intrinsics.fx,
        fy=rgb_intrinsics.fy,
        ppx=rgb_intrinsics.ppx,
        ppy=rgb_intrinsics.ppy,
        coeffs=rgb_intrinsics.coeffs,
    )

    np.savez(
        base_dir / "depth_intrinsics.npz",
        fx=depth_intrinsics.fx,
        fy=depth_intrinsics.fy,
        ppx=depth_intrinsics.ppx,
        ppy=depth_intrinsics.ppy,
        coeffs=depth_intrinsics.coeffs,
        depth_scale=depth_scale,
    )

    print(f"RGB Intrinsics: {rgb_intrinsics}")
    print(f"RGB Distortion Coefficients: {rgb_coeffs}")
    print(f"Depth Scale: {depth_scale}")
    print(f"Depth Intrinsics: {depth_intrinsics}")
    print(f"Depth Distortion Coefficients: {depth_coeffs}")


# 5. Automatic capture function

This is the ViperX version of the original `capture_images(...)` logic. It does not generate Cartesian poses and does not call `panda.move_to_pose()`. Instead, it iterates through `q_targets_rad` and sends joint targets through LeRobot.


In [ ]:
def capture_images_viperx(
    *,
    robot: ViperX,
    camera: Camera,
    q_targets_rad: np.ndarray,
    base_dir: Path,
    start_frame: int = 0,
    delay_before_shooting: float = 3.0,
    settle_time_s: float = 0.5,
) -> None:
    """Automatically collect hand-in-eye calibration samples with ViperX.

    Input:
        robot: connected LeRobot ViperX instance.
        camera: RealSense Camera wrapper, not yet started by this function's caller.
        q_targets_rad: array with shape (N, 6), radians, ordered by ViperX_ARM_JOINTS.
        base_dir: output dataset root.
        start_frame: first frame index for saved files.
        delay_before_shooting: wait after starting camera streams.
        settle_time_s: wait after each robot command before shooting.

    Output:
        No return value. Writes files under ``base_dir``:
            rgb/<frame>.png
            depth/<frame>.npy
            joints/joints_<frame>.npy
            rgb_intrinsics.npz
            depth_intrinsics.npz
    """
    camera.start()
    save_camera_intrinsics(camera, base_dir)

    # Drop one frame and wait for camera exposure/buffers.
    _, _ = camera.shoot()
    time.sleep(delay_before_shooting)

    for offset, q_target_rad in enumerate(q_targets_rad):
        frame_num = start_frame + offset
        print(f"Capturing frame {frame_num}: target q_rad={q_target_rad}")

        q_measured_rad = move_to_joint_target(
            robot,
            np.asarray(q_target_rad, dtype=float),
            settle_time_s=settle_time_s,
        )

        rgb_image, depth_image = camera.shoot()

        rgb_path = save_rgb(rgb_image, base_dir, frame_num)
        depth_path = save_depth(depth_image, base_dir, frame_num)
        joints_path = save_joints(q_measured_rad, base_dir, frame_num)

        print(f"Saved {rgb_path}")
        print(f"Saved {depth_path}")
        print(f"Saved {joints_path}")

    camera.stop()


# 6. Run

Before running this cell, confirm:

1. `viperx_port` is the correct port.
2. `q_targets_rad` contains safe ViperX joint targets.
3. `lerobot_pos_to_rad()` and `rad_to_lerobot_pos()` are implemented.

If the conversion functions are still placeholders, execution will stop with `NotImplementedError`. That is intentional: it marks the adapter boundary that must be filled for your hardware.


In [ ]:
device_serials = get_devices()
print("Selected RealSense serial number:", device_serials[0])

robot = create_viperx_robot(viperx_port)
camera = Camera(device_serials[0], rgb_resolution, depth_resolution)

try:
    robot.connect()
    capture_images_viperx(
        robot=robot,
        camera=camera,
        q_targets_rad=q_targets_rad,
        base_dir=base_dir,
        start_frame=0,
        delay_before_shooting=delay_before_shooting,
        settle_time_s=settle_time_s,
    )
finally:
    # Stop/disconnect best effort. Some failures may happen before camera.start().
    try:
        camera.stop()
    except Exception:
        pass
    try:
        robot.disconnect()
    except Exception:
        pass
